# State Space Models Mamba and the Selective Scan
---
### References
1. Gu, A., Goel, K. and Ré, C. (2022) 'Efficiently modeling long sequences with structured state spaces (S4)', *ICLR*. https://arxiv.org/abs/2111.00396
2. Gu, A. and Dao, T. (2023) 'Mamba: linear-time sequence modelling with selective state spaces', *arXiv*. https://arxiv.org/abs/2312.00752
3. Kalman, R.E. (1960) 'A new approach to linear filtering and prediction problems', *Journal of Basic Engineering*, 82(1). https://doi.org/10.1115/1.3662552
4. Dao, T. and Gu, A. (2024) 'Transformers are SSMs: generalised models and efficient algorithms through structured state space duality', *ICML*. https://arxiv.org/abs/2405.21060
5. Gu, A. et al. (2021) 'Combining recurrent, convolutional, and continuous-time models with linear state space layers', *NeurIPS*. https://arxiv.org/abs/2110.13985

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch
import warnings, os
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F

os.makedirs('figures_ssm', exist_ok=True)
torch.manual_seed(42); np.random.seed(42)

BG='#0F1117'; PANEL='#1A1D2E'; BORDER='#2D3154'
TEXT='#E8EAF6'; SUBTEXT='#8892B0'; GRID='#1E2140'
C1='#4F9CF9'; C2='#FF6B6B'; C3='#43D9AD'; C4='#FFB86C'
C5='#BD93F9'; C6='#FF79C6'; C7='#50FA7B'

plt.rcParams.update({
    'figure.facecolor':BG,'axes.facecolor':PANEL,'axes.edgecolor':BORDER,
    'axes.labelcolor':SUBTEXT,'axes.titlecolor':TEXT,'xtick.color':SUBTEXT,
    'ytick.color':SUBTEXT,'text.color':TEXT,'grid.color':GRID,
    'grid.linewidth':0.6,'font.family':'DejaVu Sans',
    'axes.spines.top':False,'axes.spines.right':False,
    'legend.facecolor':PANEL,'legend.edgecolor':BORDER,
})
def sax(ax): ax.set_facecolor(PANEL); ax.grid(True,alpha=0.15); [sp.set_edgecolor(BORDER) for sp in ax.spines.values()]
print('Setup complete.')

Setup complete.


## Figure 1 RNN vs Transformer vs SSM Architecture

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7), facecolor=BG)
fig.patch.set_facecolor(BG)
for ax in axes: ax.set_facecolor(BG); ax.axis('off'); ax.set_xlim(0,10); ax.set_ylim(0,10)

def box(ax,x,y,w,h,col,txt,alpha=0.3,fs=9.5):
    r=FancyBboxPatch((x-w/2,y-h/2),w,h,boxstyle='round,pad=0.1',facecolor=col,edgecolor=col,alpha=alpha,zorder=3,lw=2)
    g=FancyBboxPatch((x-w/2,y-h/2),w,h,boxstyle='round,pad=0.2',facecolor='none',edgecolor=col,alpha=0.15,zorder=2,lw=2.5)
    ax.add_patch(r); ax.add_patch(g)
    ax.text(x,y,txt,ha='center',va='center',fontsize=fs,color=col,fontweight='bold',zorder=4,multialignment='center',linespacing=1.3)

def arr(ax,x1,y1,x2,y2,col=SUBTEXT,lw=1.5,label=''):
    ax.annotate('',xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle='->',color=col,lw=lw),zorder=2)
    if label: ax.text((x1+x2)/2+0.25,(y1+y2)/2,label,fontsize=8.5,color=col,fontweight='bold')

# LEFT: RNN 
ax=axes[0]
ax.text(5,9.6,'RNN / LSTM',ha='center',fontsize=13,fontweight='bold',color=C1)
ax.text(5,9.0,'Sequential — O(L) steps, no parallelism',ha='center',fontsize=9,color=SUBTEXT)
# Tokens
for i,x in enumerate([1.5,3.5,5.5,7.5]):
    box(ax,x,7.2,1.5,0.7,C4,f'x_{i+1}',alpha=0.25,fs=9)
# Hidden states
for i,x in enumerate([1.5,3.5,5.5,7.5]):
    box(ax,x,5.5,1.5,0.8,C1,f'h_{i+1}',alpha=0.3,fs=9)
    arr(ax,x,6.86,x,5.9,col=C4)
for i in range(3):
    arr(ax,2.25+i*2,5.5,3.25+i*2,5.5,col=C1)
# Output
for i,x in enumerate([1.5,3.5,5.5,7.5]):
    box(ax,x,4.0,1.5,0.7,C3,f'y_{i+1}',alpha=0.25,fs=9)
    arr(ax,x,5.1,x,4.35,col=C3)

box(ax,5,2.6,8.5,0.7,C2,'✗  Sequential: token t needs h_{t-1}',alpha=0.2,fs=9)
box(ax,5,1.6,8.5,0.7,C2,'✗  Long-range: gradient vanishes over L steps',alpha=0.2,fs=9)
box(ax,5,0.7,8.5,0.7,C3,'✓  Constant memory O(d) per step',alpha=0.2,fs=9)

#  MIDDLE: Transformer
ax=axes[1]
ax.text(5,9.6,'Transformer',ha='center',fontsize=13,fontweight='bold',color=C5)
ax.text(5,9.0,'Full attention  O(L²) cost, parallel',ha='center',fontsize=9,color=SUBTEXT)
# Tokens
for i,x in enumerate([1.5,3.5,5.5,7.5]):
    box(ax,x,7.5,1.5,0.7,C4,f'x_{i+1}',alpha=0.25,fs=9)
# Attention (all-to-all)
box(ax,5,5.8,8.5,1.2,C5,'Self-Attention\n(each token attends to ALL others)',alpha=0.28,fs=9)
for i,x in enumerate([1.5,3.5,5.5,7.5]):
    arr(ax,x,7.15,x,6.4,col=C5,lw=0.8)
    arr(ax,x,5.2,x,4.55,col=C5,lw=0.8)
# Output
for i,x in enumerate([1.5,3.5,5.5,7.5]):
    box(ax,x,4.2,1.5,0.7,C3,f'y_{i+1}',alpha=0.25,fs=9)

box(ax,5,2.6,8.5,0.7,C3,'✓  Parallel: all tokens processed at once',alpha=0.2,fs=9)
box(ax,5,1.6,8.5,0.7,C3,'✓  Long-range: direct connections between any tokens',alpha=0.2,fs=9)
box(ax,5,0.7,8.5,0.7,C2,'✗  Memory/compute: O(L²) — expensive for long L',alpha=0.2,fs=9)

# RIGHT: SSM / Mamba
ax=axes[2]
ax.text(5,9.6,'SSM / Mamba',ha='center',fontsize=13,fontweight='bold',color=C3)
ax.text(5,9.0,'Linear recurrence O(L) parallel, O(1) inference',ha='center',fontsize=9,color=SUBTEXT)
# Tokens
for i,x in enumerate([1.5,3.5,5.5,7.5]):
    box(ax,x,7.5,1.5,0.7,C4,f'x_{i+1}',alpha=0.25,fs=9)
# SSM convolution
box(ax,5,5.8,8.5,1.2,C3,'SSM Selective Scan\n(input-dependent A,B,C matrices)',alpha=0.28,fs=9)
for i,x in enumerate([1.5,3.5,5.5,7.5]):
    arr(ax,x,7.15,x,6.4,col=C3,lw=0.8)
    arr(ax,x,5.2,x,4.55,col=C3,lw=0.8)
# Output
for i,x in enumerate([1.5,3.5,5.5,7.5]):
    box(ax,x,4.2,1.5,0.7,C3,f'y_{i+1}',alpha=0.25,fs=9)

box(ax,5,2.6,8.5,0.7,C3,'✓  Parallel: computed as 1D convolution at training',alpha=0.2,fs=9)
box(ax,5,1.6,8.5,0.7,C3,'✓  Efficient: O(L) compute, O(1) memory at inference',alpha=0.2,fs=9)
box(ax,5,0.7,8.5,0.7,C3,'✓  Long-range: state space captures distant dependencies',alpha=0.2,fs=9)

fig.text(0.5,1.01,'Sequence Models: RNN vs Transformer vs SSM (Mamba)',ha='center',fontsize=15,fontweight='bold',color=TEXT)
fig.text(0.5,0.975,'SSMs combine the best of both: parallel training (like transformers) + linear inference (like RNNs)',ha='center',fontsize=10.5,color=SUBTEXT)
plt.tight_layout()
plt.savefig('figures_ssm/fig1_rnn_transformer_ssm.png',dpi=180,bbox_inches='tight',facecolor=BG)
plt.show(); print('Figure 1 saved.')

Figure 1 saved.


## SSM Implementation + Figure 2 State Space Equations

In [3]:
class S4Layer(nn.Module):
    """
    Simplified S4-style SSM layer (Gu et al. 2022).
    Continuous SSM: x'(t) = Ax(t) + Bu(t), y(t) = Cx(t)
    Discretised:    x_k   = Ā x_{k-1} + B̄ u_k, y_k = C x_k
    """
    def __init__(self, d_model, d_state=16, dt_min=0.001, dt_max=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # SSM parameters (continuous-time)
        # A: (d_model, d_state) — diagonal HiPPO initialisation
        A_real = torch.arange(1, d_state+1, dtype=torch.float32)
        self.A_log = nn.Parameter(torch.log(A_real).unsqueeze(0).expand(d_model,-1).clone())
        self.B = nn.Parameter(torch.randn(d_model, d_state) * 0.01)
        self.C = nn.Parameter(torch.randn(d_model, d_state) * 0.01)
        self.D = nn.Parameter(torch.ones(d_model))  # skip connection

        # Log timestep (learnable)
        log_dt = torch.rand(d_model) * (np.log(dt_max) - np.log(dt_min)) + np.log(dt_min)
        self.log_dt = nn.Parameter(log_dt)

        self.in_proj  = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def discretise(self):
        """ZOH discretisation: Ā = exp(Δ·A), B̄ = (Ā−I)·A⁻¹·B."""
        dt  = torch.exp(self.log_dt)           # (d_model,)
        A   = -torch.exp(self.A_log)            # (d_model, d_state) negative → stable
        dA  = torch.exp(dt.unsqueeze(1) * A)    # Ā = exp(Δ·A)
        dB  = dt.unsqueeze(1) * self.B          # B̄ ≈ Δ·B  (simplified)
        return dA, dB

    def forward(self, u):
        """u: (B, L, d_model) → y: (B, L, d_model)"""
        B_sz, L, _ = u.shape
        u = self.in_proj(u)                    # (B, L, d_model)
        dA, dB = self.discretise()

        # Recurrent scan (O(L) sequential)
        x = torch.zeros(B_sz, self.d_model, self.d_state, device=u.device)
        ys = []
        for t in range(L):
            x = dA.unsqueeze(0) * x + dB.unsqueeze(0) * u[:,t,:].unsqueeze(-1)
            y = (x * self.C.unsqueeze(0)).sum(-1)  # C·x
            ys.append(y)

        y = torch.stack(ys, dim=1)             # (B, L, d_model)
        y = y + self.D.unsqueeze(0).unsqueeze(0) * u  # skip connection
        return self.out_proj(y)


class SelectiveSSM(nn.Module):
    """
    Mamba-style selective state space (Gu & Dao 2023).
    Key innovation: B, C, Δ are functions of the INPUT — input-dependent dynamics.
    """
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        d_inner = d_model * expand
        self.in_proj   = nn.Linear(d_model, d_inner * 2)
        self.conv1d    = nn.Conv1d(d_inner, d_inner, kernel_size=d_conv,
                                   padding=d_conv-1, groups=d_inner)
        # Selective projections — B, C, dt depend on input
        self.x_proj    = nn.Linear(d_inner, d_state*2 + 1)  # B, C, dt
        self.dt_proj   = nn.Linear(1, d_inner)
        self.A_log     = nn.Parameter(torch.log(torch.arange(1,d_state+1).float())
                                      .unsqueeze(0).expand(d_inner,-1).clone())
        self.D         = nn.Parameter(torch.ones(d_inner))
        self.out_proj  = nn.Linear(d_inner, d_model)
        self.d_state   = d_state; self.d_inner = d_inner

    def forward(self, u):
        B_sz, L, _ = u.shape
        xz  = self.in_proj(u)                   # (B, L, 2*d_inner)
        x,z = xz.chunk(2, dim=-1)               # gate z

        # Depthwise conv for local context
        x = self.conv1d(x.transpose(1,2))[:,:,:L].transpose(1,2)
        x = F.silu(x)

        # Selective: compute B, C, dt from input
        BCdt = self.x_proj(x)                   # (B, L, 2N+1)
        B_sel = BCdt[..., :self.d_state]
        C_sel = BCdt[..., self.d_state:2*self.d_state]
        dt    = F.softplus(self.dt_proj(BCdt[..., -1:])).squeeze(-1)  # (B, L, d_inner)

        # Discretise A per-token (input-dependent!)
        A   = -torch.exp(self.A_log)             # (d_inner, N)
        dA  = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))
        dB  = dt.unsqueeze(-1) * B_sel.unsqueeze(2)  # simplified

        # Selective scan
        state = torch.zeros(B_sz, self.d_inner, self.d_state, device=u.device)
        ys = []
        for t in range(L):
            state = dA[:,t] * state + dB[:,t] * x[:,t].unsqueeze(-1)
            y_t   = (state * C_sel[:,t].unsqueeze(1)).sum(-1)
            ys.append(y_t)

        y = torch.stack(ys,1) + self.D * x       # (B,L,d_inner)
        y = y * F.silu(z)
        return self.out_proj(y)

print('S4Layer and SelectiveSSM defined.')

S4Layer and SelectiveSSM defined.


## Figure 2 State Space Equations and Discretisation

In [ ]:
fig = plt.figure(figsize=(17, 7), facecolor=BG)
gs  = GridSpec(1, 3, figure=fig, wspace=0.18)
fig.patch.set_facecolor(BG)

#  Panel 1: Continuous SSM
ax1=fig.add_subplot(gs[0]); ax1.set_facecolor(PANEL); ax1.axis('off'); ax1.set_xlim(0,10); ax1.set_ylim(0,10)
ax1.text(5,9.5,'Continuous SSM',ha='center',fontsize=12,fontweight='bold',color=C1)
ax1.text(5,8.9,'From control theory (Kalman 1960)',ha='center',fontsize=9,color=SUBTEXT)

boxes_c=[
    (5,7.8,"x'(t) = A x(t) + B u(t)",C1,"State transition"),
    (5,6.4,"y(t) = C x(t) + D u(t)",C3,"Output equation"),
    (5,5.0,"u(t) → input signal",C4,""),
    (5,4.2,"x(t) ∈ ℝᴺ → hidden state",C5,""),
    (5,3.4,"y(t) → output",C3,""),
]
for cx,cy,txt,col,sub in boxes_c:
    r=FancyBboxPatch((0.5,cy-0.38),9.0,0.75,boxstyle='round,pad=0.08',facecolor=col,edgecolor=col,alpha=0.2,zorder=3)
    ax1.add_patch(r)
    ax1.text(cx,cy+(0.12 if sub else 0),txt,ha='center',va='center',fontsize=10,color=col,fontweight='bold',zorder=4)
    if sub: ax1.text(cx,cy-0.2,sub,ha='center',va='center',fontsize=8,color=SUBTEXT,zorder=4)

ax1.text(5,2.2,'A: state matrix (N×N)',ha='center',fontsize=8.5,color=SUBTEXT)
ax1.text(5,1.7,'B: input matrix (N×1)',ha='center',fontsize=8.5,color=SUBTEXT)
ax1.text(5,1.2,'C: output matrix (1×N)',ha='center',fontsize=8.5,color=SUBTEXT)
ax1.text(5,0.7,'N = state dimension (d_state)',ha='center',fontsize=8.5,color=C5)

#  Panel 2: Discretisation 
ax2=fig.add_subplot(gs[1]); ax2.set_facecolor(PANEL); ax2.axis('off'); ax2.set_xlim(0,10); ax2.set_ylim(0,10)
ax2.text(5,9.5,'ZOH Discretisation',ha='center',fontsize=12,fontweight='bold',color=C4)
ax2.text(5,8.9,'Continuous → discrete with step size Δ',ha='center',fontsize=9,color=SUBTEXT)

disc_items=[
    (5,7.8,"Ā = exp(Δ·A)",C4,"Discrete state matrix"),
    (5,6.3,"B̄ = (Ā−I)·A⁻¹·B",C4,"Discrete input matrix"),
    (5,4.8,"x_k = Ā x_{k-1} + B̄ u_k",C1,"Recurrent form (inference)"),
    (5,3.5,"y_k = C x_k + D u_k",C3,"Output at step k"),
]
for cx,cy,txt,col,sub in disc_items:
    r=FancyBboxPatch((0.5,cy-0.42),9.0,0.82,boxstyle='round,pad=0.08',facecolor=col,edgecolor=col,alpha=0.22,zorder=3)
    ax2.add_patch(r)
    ax2.text(cx,cy+0.12,txt,ha='center',va='center',fontsize=10,color=col,fontweight='bold',zorder=4)
    ax2.text(cx,cy-0.2,sub,ha='center',va='center',fontsize=8,color=SUBTEXT,zorder=4)

ax2.text(5,2.2,'Δ = timestep (learnable per feature)',ha='center',fontsize=9,color=C4,fontweight='bold')
ax2.text(5,1.6,'Key: Δ controls how fast the state evolves',ha='center',fontsize=8.5,color=SUBTEXT)
ax2.text(5,1.0,'Large Δ: more focused on current input',ha='center',fontsize=8.5,color=SUBTEXT)
ax2.text(5,0.5,'Small Δ: more memory of past context',ha='center',fontsize=8.5,color=SUBTEXT)

# ── Panel 3: Two computation modes ──
ax3=fig.add_subplot(gs[2]); ax3.set_facecolor(PANEL); ax3.axis('off'); ax3.set_xlim(0,10); ax3.set_ylim(0,10)
ax3.text(5,9.5,'Two Computation Modes',ha='center',fontsize=12,fontweight='bold',color=C3)
ax3.text(5,8.9,'SSMs are uniquely both RNN and CNN',ha='center',fontsize=9,color=SUBTEXT)

r1=FancyBboxPatch((0.3,6.2),9.4,2.4,boxstyle='round,pad=0.15',facecolor=C1,edgecolor=C1,alpha=0.18,zorder=3)
ax3.add_patch(r1)
ax3.text(5,8.0,'Training — Convolutional mode',ha='center',fontsize=10,color=C1,fontweight='bold',zorder=4)
ax3.text(5,7.4,'y = u * K (global convolution)',ha='center',fontsize=9.5,color=TEXT,zorder=4)
ax3.text(5,6.8,'K = [CB̄, CĀB̄, CĀ²B̄, ...]  (kernel)',ha='center',fontsize=9,color=SUBTEXT,zorder=4)
ax3.text(5,6.4,'✓ Fully parallel — O(L log L) with FFT',ha='center',fontsize=9,color=C3,zorder=4)

r2=FancyBboxPatch((0.3,3.5),9.4,2.4,boxstyle='round,pad=0.15',facecolor=C3,edgecolor=C3,alpha=0.18,zorder=3)
ax3.add_patch(r2)
ax3.text(5,5.3,'Inference — Recurrent mode',ha='center',fontsize=10,color=C3,fontweight='bold',zorder=4)
ax3.text(5,4.7,'x_k = Ā x_{k-1} + B̄ u_k',ha='center',fontsize=9.5,color=TEXT,zorder=4)
ax3.text(5,4.1,'One step at a time, constant state',ha='center',fontsize=9,color=SUBTEXT,zorder=4)
ax3.text(5,3.7,'✓ O(1) memory — no KV cache growth',ha='center',fontsize=9,color=C3,zorder=4)

r3=FancyBboxPatch((0.3,1.0),9.4,2.2,boxstyle='round,pad=0.15',facecolor=C5,edgecolor=C5,alpha=0.18,zorder=3)
ax3.add_patch(r3)
ax3.text(5,2.7,'Same parameters — dual view',ha='center',fontsize=10,color=C5,fontweight='bold',zorder=4)
ax3.text(5,2.1,'Training uses conv for speed',ha='center',fontsize=9,color=TEXT,zorder=4)
ax3.text(5,1.5,'Inference uses recurrence for memory',ha='center',fontsize=9,color=SUBTEXT,zorder=4)

fig.text(0.5,1.01,'State Space Model — Equations, Discretisation, and Dual Computation',ha='center',fontsize=15,fontweight='bold',color=TEXT)
fig.text(0.5,0.975,"SSMs bridge continuous control theory and deep learning — trained as CNNs, deployed as RNNs",ha='center',fontsize=10.5,color=SUBTEXT)
plt.tight_layout()
plt.savefig('figures_ssm/fig2_ssm_equations.png',dpi=180,bbox_inches='tight',facecolor=BG)
plt.show(); print('Figure 2 saved.')

Figure 2 saved.


## Figure 3 Selective Scan: The Mamba Innovation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), gridspec_kw={'wspace': 0.15})
fig.patch.set_facecolor(BG)
for ax in axes: ax.set_facecolor(BG); ax.axis('off'); ax.set_xlim(0,12); ax.set_ylim(0,10)

def sbox(ax,x,y,w,h,col,txt,alpha=0.28,fs=9):
    r=FancyBboxPatch((x-w/2,y-h/2),w,h,boxstyle='round,pad=0.1',facecolor=col,edgecolor=col,alpha=alpha,zorder=3,lw=2)
    g=FancyBboxPatch((x-w/2,y-h/2),w,h,boxstyle='round,pad=0.22',facecolor='none',edgecolor=col,alpha=0.14,zorder=2,lw=3)
    ax.add_patch(r); ax.add_patch(g)
    ax.text(x,y,txt,ha='center',va='center',fontsize=fs,color=col,fontweight='bold',zorder=4,multialignment='center',linespacing=1.3)

#  LEFT: S4 fixed parameters
ax=axes[0]
ax.text(6,9.6,'S4 — Fixed State Space Parameters',ha='center',fontsize=12,fontweight='bold',color=C1)
ax.text(6,9.1,'A, B, C are constant — same dynamics for every input',ha='center',fontsize=9,color=SUBTEXT)

tokens=['"The"','"cat"','"sat"','"on"']
for i,tok in enumerate(tokens):
    x=1.5+i*2.5
    sbox(ax,x,7.8,2.0,0.7,C4,tok,fs=9)
    sbox(ax,x,6.0,2.0,0.7,C1,'A,B,C\nfixed',fs=7.5)
    ax.annotate('',xy=(x,6.35),xytext=(x,7.45),arrowprops=dict(arrowstyle='->',color=SUBTEXT,lw=1.2))

sbox(ax,6,4.2,10,0.85,C1,'Same (Ā,B̄,C) for ALL tokens',fs=9)
ax.text(6,3.3,'✗  Cannot selectively focus on important tokens',ha='center',fontsize=9.5,color=C2,fontweight='bold')
ax.text(6,2.7,'✗  "The" and "cat" get identical dynamics',ha='center',fontsize=9,color=SUBTEXT)
ax.text(6,2.1,'✗  Hard to ignore irrelevant context',ha='center',fontsize=9,color=SUBTEXT)
sbox(ax,6,1.2,10,0.7,C2,'Like a convolutional filter — position-independent',alpha=0.2,fs=9)

# RIGHT: Mamba selective 
ax=axes[1]
ax.text(6,9.6,'Mamba — Selective State Space',ha='center',fontsize=12,fontweight='bold',color=C3)
ax.text(6,9.1,'B, C, Δ depend on INPUT — different dynamics per token',ha='center',fontsize=9,color=SUBTEXT)

select_cols=[C4,C2,C3,SUBTEXT]  # important tokens get brighter
dt_vals=['Δ=small','Δ=LARGE','Δ=LARGE','Δ=tiny']  # cat and sat are important
for i,(tok,sc,dt) in enumerate(zip(tokens,select_cols,dt_vals)):
    x=1.5+i*2.5
    sbox(ax,x,7.8,2.0,0.7,sc,tok,fs=9)
    sbox(ax,x,6.2,2.0,0.9,sc,f'B(x), C(x)\n{dt}',alpha=0.3,fs=7.5)
    ax.annotate('',xy=(x,6.65),xytext=(x,7.45),arrowprops=dict(arrowstyle='->',color=sc,lw=1.5))

sbox(ax,6,4.5,10,0.85,C3,'Input-dependent (Ā,B̄,C) — per token',fs=9)
ax.text(6,3.5,'✓  Focus selectively: large Δ for important tokens',ha='center',fontsize=9.5,color=C3,fontweight='bold')
ax.text(6,2.9,'✓  Forget selectively: small Δ resets the state',ha='center',fontsize=9,color=SUBTEXT)
ax.text(6,2.3,'✓  Content-aware: like attention, but O(L) not O(L²)',ha='center',fontsize=9,color=SUBTEXT)
sbox(ax,6,1.2,10,0.7,C3,'✓  The key innovation of Mamba (Gu & Dao 2023)',alpha=0.25,fs=9)

fig.text(0.5,1.01,'S4 vs Mamba — Fixed vs Selective State Space',ha='center',fontsize=15,fontweight='bold',color=TEXT)
fig.text(0.5,0.975,'Mamba makes B, C, Δ functions of the input — enabling content-aware, selective memory',ha='center',fontsize=10.5,color=SUBTEXT)
plt.tight_layout()
plt.savefig('figures_ssm/fig3_selective_scan.png',dpi=180,bbox_inches='tight',facecolor=BG)
plt.show(); print('Figure 3 saved.')

Figure 3 saved.


## Figure 4 SSM on Sequence Classification + HiPPO Init

In [ ]:
#  Train S4 and compare to LSTM on a synthetic task 
# Task: classify whether a signal contains a specific frequency pattern
def make_seq_dataset(n=500, L=64, seed=42):
    np.random.seed(seed)
    t = np.linspace(0, 4*np.pi, L)
    X, y = [], []
    for _ in range(n):
        cls = np.random.randint(0,2)
        freq = 2.0 if cls==0 else 5.0
        sig = np.sin(freq*t) + np.random.randn(L)*0.4
        X.append(sig); y.append(cls)
    return np.array(X,dtype=np.float32), np.array(y)

X_seq, y_seq = make_seq_dataset(n=800, L=64)
X_tr,X_te,y_tr,y_te = X_seq[:600],X_seq[600:],y_seq[:600],y_seq[600:]
X_tr_t=torch.FloatTensor(X_tr).unsqueeze(-1)
y_tr_t=torch.LongTensor(y_tr)
X_te_t=torch.FloatTensor(X_te).unsqueeze(-1)
y_te_t=torch.LongTensor(y_te)

# Simple SSM classifier
class SSMClassifier(nn.Module):
    def __init__(self, d_model=16, d_state=8):
        super().__init__()
        self.ssm = S4Layer(d_model, d_state)
        self.inp = nn.Linear(1, d_model)
        self.head= nn.Linear(d_model,2)
    def forward(self,x):
        x=self.inp(x); x=self.ssm(x); return self.head(x.mean(1))

class LSTMClassifier(nn.Module):
    def __init__(self, d_model=16):
        super().__init__()
        self.lstm=nn.LSTM(1,d_model,batch_first=True)
        self.head=nn.Linear(d_model,2)
    def forward(self,x):
        _,(h,_)=self.lstm(x); return self.head(h.squeeze(0))

results_train={'SSM':[],'LSTM':[]}
results_test ={'SSM':[],'LSTM':[]}

for name,Model in [('SSM',SSMClassifier),('LSTM',LSTMClassifier)]:
    model=Model(); opt=torch.optim.Adam(model.parameters(),lr=3e-3)
    for ep in range(200):
        model.train(); opt.zero_grad()
        loss=F.cross_entropy(model(X_tr_t),y_tr_t); loss.backward(); opt.step()
        if ep%20==0:
            model.eval()
            with torch.no_grad():
                tr_acc=(model(X_tr_t).argmax(1)==y_tr_t).float().mean().item()
                te_acc=(model(X_te_t).argmax(1)==y_te_t).float().mean().item()
            results_train[name].append(tr_acc*100)
            results_test[name].append(te_acc*100)
    model.eval()
    with torch.no_grad(): final=( model(X_te_t).argmax(1)==y_te_t).float().mean()
    print(f'{name} final test acc: {final*100:.1f}%')

# HiPPO visualisation
def hippo_matrix(N):
    """HiPPO-LegS matrix — optimal polynomial basis for function memorisation."""
    A = np.zeros((N,N))
    for n in range(N):
        for k in range(n):
            A[n,k] = -np.sqrt((2*n+1)*(2*k+1))
        A[n,n] = -(n+1)
    return A

fig, axes = plt.subplots(1,3,figsize=(18,5.8),gridspec_kw={'wspace':0.3})
fig.patch.set_facecolor(BG)

# Panel 1: training curves
ax=axes[0]; sax(ax)
eps=np.arange(0,200,20)
for name,col,ls in [('SSM',C3,'-'),('LSTM',C1,'--')]:
    ax.plot(eps,results_test[name],ls,color=col,lw=2.5,ms=6,marker='o',
            markeredgecolor='white',mew=0.7,label=f'{name} test',
            path_effects=[pe.withStroke(linewidth=5,foreground=col+'44')])
ax.set_xlabel('Epoch',color=SUBTEXT,fontsize=11); ax.set_ylabel('Test accuracy (%)',color=SUBTEXT,fontsize=11)
ax.set_title('SSM vs LSTM\nFrequency classification task',fontsize=11,fontweight='bold',color=TEXT,pad=8)
ax.legend(fontsize=10); ax.set_ylim(45,105)

# Panel 2: HiPPO matrix visualisation
ax2=axes[1]; ax2.set_facecolor(PANEL)
H=hippo_matrix(16)
cmap2=LinearSegmentedColormap.from_list('hippo',['#0F1117','#1a2a6b','#4F9CF9','#FF6B6B'],N=256)
im=ax2.imshow(H,cmap=cmap2,aspect='auto')
plt.colorbar(im,ax=ax2,label='Matrix value')
ax2.set_xlabel('State index k',color=SUBTEXT,fontsize=10)
ax2.set_ylabel('State index n',color=SUBTEXT,fontsize=10)
ax2.set_title('HiPPO-LegS Matrix (N=16)\nOptimal A initialisation for long-range memory',fontsize=10,fontweight='bold',color=TEXT,pad=8)
[sp.set_edgecolor(BORDER) for sp in ax2.spines.values()]

# Panel 3: efficiency comparison
ax3=axes[2]; sax(ax3)
L_vals=[64,256,1024,4096,16384]
rnn_cost   =[L for L in L_vals]                  # O(L)
transformer=[L**2/1000 for L in L_vals]           # O(L²)
ssm_train  =[L*np.log2(L)/1000 for L in L_vals]  # O(L log L) conv
ssm_infer  =[1]*len(L_vals)                        # O(1)

for vals,col,label,ls in [
    (transformer,C2,'Transformer O(L²)','-'),
    (rnn_cost,C1,'RNN O(L)','-'),
    (ssm_train,C3,'SSM train O(L log L)','--'),
    (ssm_infer,C4,'SSM infer O(1)','-.'),
]:
    ax3.semilogy(L_vals,vals,ls,color=col,lw=2.5,label=label,
                 path_effects=[pe.withStroke(linewidth=5,foreground=col+'44')])
ax3.set_xlabel('Sequence length L',color=SUBTEXT,fontsize=11)
ax3.set_ylabel('Relative cost (log scale)',color=SUBTEXT,fontsize=11)
ax3.set_title('Compute Scaling with Sequence Length\nSSM inference is O(1) — constant per step',fontsize=10,fontweight='bold',color=TEXT,pad=8)
ax3.legend(fontsize=9)

fig.text(0.5,1.01,'SSM Performance, HiPPO Initialisation, and Scaling',ha='center',fontsize=15,fontweight='bold',color=TEXT)
fig.text(0.5,0.975,'SSMs match LSTM accuracy while scaling far more efficiently to long sequences',ha='center',fontsize=10.5,color=SUBTEXT)
plt.savefig('figures_ssm/fig4_ssm_results.png',dpi=180,bbox_inches='tight',facecolor=BG)
plt.show(); print('Figure 4 saved.')

SSM final test acc: 75.5%


LSTM final test acc: 100.0%


Figure 4 saved.


## Figure 5 Mamba Block Architecture

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7.5), facecolor=BG)
fig.patch.set_facecolor(BG)
for ax in axes: ax.set_facecolor(BG); ax.axis('off'); ax.set_xlim(0,10); ax.set_ylim(0,10)

def mbox(ax,x,y,w,h,col,txt,alpha=0.28,fs=9):
    r=FancyBboxPatch((x-w/2,y-h/2),w,h,boxstyle='round,pad=0.1',facecolor=col,edgecolor=col,alpha=alpha,zorder=3,lw=2)
    g=FancyBboxPatch((x-w/2,y-h/2),w,h,boxstyle='round,pad=0.22',facecolor='none',edgecolor=col,alpha=0.14,zorder=2,lw=2.5)
    ax.add_patch(r); ax.add_patch(g)
    ax.text(x,y,txt,ha='center',va='center',fontsize=fs,color=col,fontweight='bold',zorder=4,multialignment='center',linespacing=1.3)

def marr(ax,x1,y1,x2,y2,col=SUBTEXT,lw=1.5):
    ax.annotate('',xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle='->',color=col,lw=lw),zorder=2)

# LEFT: Mamba block
ax=axes[0]
ax.text(5,9.6,'Mamba Block Architecture',ha='center',fontsize=12,fontweight='bold',color=C3)

components=[
    (5,8.7,4.5,0.7,TEXT,'Input x',0.15,9.5),
    (5,7.5,6.5,0.8,C1,'Linear expand (×2)',0.28,9.5),
    (2.2,6.2,3.0,0.75,C4,'Depthwise Conv1D\n(local context)',0.28,9),
    (2.2,5.0,3.0,0.75,C4,'SiLU activation',0.28,9),
    (2.2,3.8,3.0,0.75,C3,'SSM Selective Scan\n(B,C,Δ from input)',0.3,9),
    (7.8,6.2,3.0,0.75,C5,'Gate branch\n(SiLU)',0.28,9),
    (5,2.6,6.5,0.75,C3,'Element-wise multiply (gating)',0.28,9.5),
    (5,1.5,4.5,0.7,C1,'Linear project out',0.28,9.5),
    (5,0.5,4.5,0.7,TEXT,'Output y',0.15,9.5),
]
for cx,cy,w,h,col,txt,alpha,fs in components:
    mbox(ax,cx,cy,w,h,col,txt,alpha,fs)

# Arrows
marr(ax,5,8.35,5,7.9,C1)
marr(ax,3.75,7.1,2.2,6.58,C4)
marr(ax,6.25,7.1,7.8,6.58,C5)
marr(ax,2.2,5.85,2.2,5.38,C4)
marr(ax,2.2,4.63,2.2,4.18,C3)
marr(ax,2.2,3.43,3.5,2.98,C3)
marr(ax,7.8,5.83,6.5,2.98,C5)
marr(ax,5,2.23,5,1.88,C1)
marr(ax,5,1.13,5,0.85,TEXT)

# Residual
ax.annotate('',xy=(9.2,0.5),xytext=(9.2,8.7),arrowprops=dict(arrowstyle='->',color=SUBTEXT,lw=1.2,linestyle='dashed'),zorder=2)
ax.text(9.5,4.6,'Skip\nconn.',ha='center',fontsize=8,color=SUBTEXT,rotation=90)

#RIGHT: Comparison table
ax2=axes[1]
ax2.text(5,9.6,'SSM Model Comparison',ha='center',fontsize=12,fontweight='bold',color=TEXT)

rows=[
    ('Property',    'S4',       'Mamba',     'Transformer', SUBTEXT),
    ('A matrix',    'Fixed\nHiPPO','Fixed',  'N/A',         C1),
    ('B matrix',    'Fixed',    'Input-dep.','N/A',         C3),
    ('C matrix',    'Fixed',    'Input-dep.','N/A',         C4),
    ('Δ timestep',  'Fixed',    'Input-dep.','N/A',         C5),
    ('Train cost',  'O(L logL)','O(L logL)', 'O(L²)',       C2),
    ('Infer cost',  'O(1)/step','O(1)/step', 'O(L²)',       C3),
    ('Memory',      'O(N)',     'O(N)',      'O(L)',         C3),
    ('Content\naware','No',     'Yes',       'Yes',         C3),
]
row_h=0.82; start_y=8.7
col_xs=[1.5,4.0,6.3,8.8]
row_cols=[C1,C3,C4,C5]
for ri,(prop,s4,mamba,trans,hcol) in enumerate(rows):
    y=start_y-ri*row_h
    bg=PANEL if ri%2==0 else '#1f2340'
    r=FancyBboxPatch((0.2,y-0.35),9.6,0.7,boxstyle='round,pad=0.05',facecolor=bg,edgecolor=BORDER,alpha=1.0,zorder=2)
    ax2.add_patch(r)
    for xi,txt,tcol in [(0,prop,hcol),(1,s4,C1),(2,mamba,C3),(3,trans,C5)]:
        ax2.text(col_xs[xi],y,txt,ha='center',va='center',fontsize=8.5 if ri==0 else 8,
                 color=tcol if ri>0 else SUBTEXT,fontweight='bold' if ri==0 else 'normal',
                 zorder=3,multialignment='center',linespacing=1.2)

fig.text(0.5,1.01,'Mamba Block and SSM Model Comparison',ha='center',fontsize=15,fontweight='bold',color=TEXT)
fig.text(0.5,0.975,'Mamba achieves input-dependent selectivity while maintaining O(1) inference memory',ha='center',fontsize=10.5,color=SUBTEXT)
plt.tight_layout()
plt.savefig('figures_ssm/fig5_mamba_architecture.png',dpi=180,bbox_inches='tight',facecolor=BG)
plt.show(); print('Figure 5 saved.')

Figure 5 saved.


## Summary

| Model | A,B,C | Train | Infer | Memory | Content-aware |
|-------|-------|-------|-------|--------|---------------|
| S4 | Fixed | O(L log L) | O(1) | O(N) | No |
| Mamba | Input-dep. | O(L log L) | O(1) | O(N) | Yes |
| Transformer | N/A | O(L²) | O(L²) | O(L) | Yes |
| LSTM | N/A | O(L) | O(L) | O(d) | Partial |